# OCR Bilans Fiscaux Algériens — V15 — JSON + contrôles + liasse Excel

> V15 : postes groupés (sous-totaux) intégrés au schéma + au prompt + aux formules de contrôle.
> Règle : un poste groupé **absent** (null) → **aucun écart** ; un écart n'est calculé que si la valeur est extraite.

## Règle d'or
Ne jamais inventer de valeur : absent/illisible → null.

In [ ]:
%pip install -q -U 'transformers>=4.57.0' accelerate pymupdf pillow psutil
print('✅ Dépendances OK')

In [ ]:
import time, json, re, gc, copy, unicodedata
import numpy as np
from pathlib import Path
from datetime import datetime
from collections import Counter
import fitz, torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText
print('✅ Imports OK')

In [ ]:
MODEL_PATH = '/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen3.6-27B-FP8/main'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MAX_NEW_TOKENS = 4096
IMAGE_MAX_SIZE = 2024
MIN_PIXELS = 4*32*32
MAX_PIXELS = 2000*32*32
PDF_ZOOM = 3.0
BLANK_THRESHOLD = 0.95
CLASSIF_BATCH_SIZE = 16
GPU_BATCH_SIZE = 4
INPUT_DIR = Path('/mnt/Risk/bilans_in')
OUTPUT_DIR = Path('/mnt/Risk/bilans_out')
JSON_DIR = OUTPUT_DIR / 'json_bilans_v15'
LOG_PATH = OUTPUT_DIR / 'pipeline_bilans_v15.log'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
JSON_DIR.mkdir(parents=True, exist_ok=True)
pdfs = sorted(INPUT_DIR.glob('*.pdf'))
print('Device:', DEVICE, '| Dossiers:', len(pdfs))

In [ ]:
def log(msg):
    ligne = datetime.now().strftime('%Y-%m-%d %H:%M:%S') + ' — ' + str(msg)
    print(ligne, flush=True)
    with open(LOG_PATH, 'a', encoding='utf-8') as f: f.write(ligne + chr(10))
print('✅ Log OK')

In [ ]:
t0 = time.time()
processor = AutoProcessor.from_pretrained(MODEL_PATH, trust_remote_code=True, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)
processor.tokenizer.padding_side = 'left'
try:
    from transformers.integrations.finegrained_fp8 import FineGrainedFP8Config as FP8Config
except ImportError:
    from transformers import FineGrainedFP8Config as FP8Config
model = AutoModelForImageTextToText.from_pretrained(MODEL_PATH, dtype=torch.bfloat16, device_map='auto', trust_remote_code=True, low_cpu_mem_usage=True, quantization_config=FP8Config(dequantize=True))
model.eval()
print('✅ Modèle chargé en ' + str(round(time.time()-t0,1)) + 's')

In [ ]:
def resize(img, max_side=IMAGE_MAX_SIZE):
    w,h = img.size
    if max(w,h) <= max_side: return img
    r = max_side/max(w,h)
    return img.resize((int(w*r), int(h*r)), Image.LANCZOS)
def strip_accents(s):
    return ''.join(c for c in unicodedata.normalize('NFKD', str(s)) if not unicodedata.combining(c))
def norm_key(s):
    s = strip_accents(str(s)).lower()
    return re.sub('[^a-z0-9]+',' ', s).strip(' ')
def estimate_skew(img):
    small = img.convert('L').copy(); small.thumbnail((500,500))
    def score(a):
        r = np.array(small.rotate(a, expand=True, fillcolor=255)) < 128
        return float(((r.sum(axis=1))**2).sum())
    best = max(range(-12,13,2), key=score)
    best = max([best-1,best-0.5,best,best+0.5,best+1], key=score)
    return best if abs(best) >= 1 else 0.0
def deskew(img):
    a = estimate_skew(img)
    if a: img = img.rotate(a, expand=True, fillcolor=(255,255,255), resample=Image.BICUBIC)
    return img
def is_blank(image, threshold=BLANK_THRESHOLD):
    arr = np.array(image.convert('L'))
    return (arr > 240).sum()/arr.size >= threshold
def pdf_to_pages(path, zoom=PDF_ZOOM):
    doc = fitz.open(path); matrix = fitz.Matrix(zoom, zoom); pages = []
    for i in range(len(doc)):
        pix = doc.load_page(i).get_pixmap(matrix=matrix, alpha=False)
        img = Image.frombytes('RGB', [pix.width, pix.height], pix.samples)
        pages.append({'index': i, 'image': resize(deskew(img))})
    doc.close()
    return pages
def parse_json(text):
    try:
        m = re.search(r'\{.*\}', text or '', re.S)
        return json.loads(m.group()) if m else {}
    except Exception:
        return {}
def apply_template(messages):
    try:
        return processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        return processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
def _decode(out_i, in_len):
    return processor.decode(out_i[in_len:], skip_special_tokens=True, clean_up_tokenization_spaces=False)
def ask_single(prompt, image):
    msgs = [{'role':'user','content':[{'type':'image','image':image},{'type':'text','text':prompt}]}]
    inputs = processor(text=[apply_template(msgs)], images=[image], return_tensors='pt').to(DEVICE)
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, repetition_penalty=1.0, pad_token_id=processor.tokenizer.eos_token_id)
    torch.cuda.synchronize()
    return {'text': _decode(out[0], inputs['input_ids'].shape[1]), 'tokens_in': int(inputs['input_ids'].shape[1]), 'tokens_out': int(out[0].shape[0]-inputs['input_ids'].shape[1]), 'elapsed': round(time.time()-t0,2)}
def ask_batch(prompt, images):
    if not images: return []
    if len(images) == 1: return [ask_single(prompt, images[0])]
    msgs = [[{'role':'user','content':[{'type':'image','image':img},{'type':'text','text':prompt}]}] for img in images]
    inputs = processor(text=[apply_template(m) for m in msgs], images=images, return_tensors='pt', padding=True).to(DEVICE)
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, repetition_penalty=1.0, pad_token_id=processor.tokenizer.eos_token_id)
    torch.cuda.synchronize()
    el = time.time()-t0; in_len = inputs['input_ids'].shape[1]; attn = inputs.get('attention_mask')
    return [{'text': _decode(out[i], in_len), 'tokens_in': int(attn[i].sum().item()) if attn is not None else in_len, 'tokens_out': int(out[i].shape[0]-in_len), 'elapsed': round(el/len(images),2)} for i in range(len(images))]
print('✅ Utilitaires OK')

In [ ]:
# SCHEMAS : V15 ajoute les sous-totaux ACTIF (postes groupés parfois présents sur les scans)
SCHEMAS = {
 'ACTIF': {'cols': ['montant_brut','amortissements_provisions_pertes','net_n','net_n1'], 'postes': {
   'ecarts_acquisition_goodwill': 'Ecart d acquisition goodwill',
   'immobilisations_incorporelles': 'Immobilisations incorporelles',
   'immobilisations_corporelles': 'Immobilisations corporelles (sous-total)',
   'terrains': 'Terrains', 'batiments': 'Batiments',
   'autres_immobilisations_corporelles': 'Autres Immobilisations corporelles',
   'immobilisations_en_concession': 'Immobilisations en concession',
   'immobilisations_en_cours': 'Immobilisations en cours',
   'immobilisations_financieres': 'Immobilisations financieres (sous-total)',
   'titres_mis_en_equivalence': 'Titres mis en equivalence',
   'autres_participations_creances': 'Autres participations et creances rattachees',
   'autres_titres_immobilises': 'Autres titres immobilises',
   'prets_actifs_financiers_non_courants': 'Prets et autres actifs financiers non courants',
   'impots_differes_actif': 'Impots Differes Actif',
   'total_actif_non_courant': 'TOTAL ACTIF NON COURANT',
   'stocks_encours': 'Stocks et encours',
   'creances_et_emplois_assimiles': 'Creances et emplois assimiles (sous-total)',
   'clients': 'Clients', 'autres_debiteurs': 'Autres debiteurs',
   'impots_assimiles_actif': 'Impots et assimiles',
   'autres_creances_assimiles': 'Autres Creances et Emplois assimiles',
   'disponibilites_et_assimiles': 'Disponibilites et assimiles (sous-total)',
   'placements_financiers_courants': 'Placements et autres actifs financiers courants',
   'tresorerie_actif': 'Tresorerie',
   'total_actif_courant': 'TOTAL ACTIF COURANT',
   'total_general_actif': 'TOTAL GENERAL ACTIF'}},
 'PASSIF': {'cols': ['n','n1'], 'postes': {
   'capital_emis': 'Capital emis', 'capital_non_appele': 'Capital non appele',
   'primes_reserves': 'Primes et reserves', 'ecart_reevaluation': 'Ecart de reevaluation',
   'ecart_equivalence': 'Ecart d equivalence', 'resultat_net_passif': 'Resultat net',
   'report_a_nouveau': 'Report a nouveau', 'part_societe_consolidante': 'Part de la societe consolidante',
   'part_minoritaires': 'Part des minoritaires', 'total_capitaux_propres': 'TOTAL I',
   'emprunts_dettes_financieres': 'Emprunts et dettes financieres',
   'impots_differes_provisionnes': 'Impots (differes et provisionnes)',
   'autres_dettes_non_courantes': 'Autres dettes non courantes',
   'provisions_produits_avance': 'Provisions et produits constatés d avance',
   'total_passifs_non_courants': 'TOTAL II',
   'fournisseurs_rattaches': 'Fournisseurs et comptes rattaches', 'impots_passif': 'Impots',
   'autres_dettes': 'Autres dettes', 'tresorerie_passif': 'Tresorerie Passif',
   'total_passifs_courants': 'TOTAL III', 'total_general_passif': 'TOTAL GENERAL PASSIF'}},
 'TCR': {'cols': ['n_debit','n_credit','n1_debit','n1_credit'], 'postes': {
   'ventes_marchandises': 'Ventes de Marchandises', 'produits_fabriques': 'Produits Fabriques',
   'prestations_services': 'Prestations de Services', 'ventes_travaux': 'Ventes de Travaux',
   'produits_annexes': 'Produits Annexes', 'rabais_remises_ristournes_accordes': 'Rabais remises ristournes accordes',
   'chiffre_affaires_net': 'Chiffre d affaires net', 'production_stockee_destockee': 'Production Stockee ou destockee',
   'production_immobilisee': 'Production immobilisee', 'subvention_exploitation': 'Subvention d exploitation',
   'production_exercice': 'I-Production de l exercice', 'achats_marchandises_vendues': 'Achats de Marchandises vendues',
   'matieres_premieres': 'Matieres premieres', 'autres_approvisionnements': 'Autres Approvisionnements',
   'variation_stocks': 'Variation des Stocks', 'achats_etudes_prestations': 'Achats d Etudes et de Prestations de services',
   'autres_consommations': 'Autres consommations', 'sous_traitance_generale': 'Sous-traitance generale',
   'locations': 'Locations', 'entretien_reparations': 'Entretien reparations et maintenance',
   'primes_assurances': 'Primes d assurances', 'personnel_exterieur': 'Personnel exterieur a l entreprise',
   'remuneration_intermediaires': 'Remuneration d intermediaires et honoraires', 'publicite': 'Publicite',
   'deplacements_missions': 'Deplacements missions et receptions', 'autres_services': 'Autres services',
   'consommations_exercice': 'II-Consommations de l exercice', 'valeur_ajoutee_exploitation': 'III-Valeur ajoutee d exploitation',
   'charges_personnel': 'Charges de personnel', 'impots_taxes_assimiles': 'Impots et taxes et versements assimiles',
   'excedent_brut_exploitation': 'IV-Excedent brut d exploitation', 'autres_produits_operationnels': 'Autres produits operationnels',
   'autres_charges_operationnelles': 'Autres charges operationnelles', 'dotations_amortissements': 'Dotations aux amortissements',
   'provisions': 'Provisions', 'pertes_valeur': 'Perte de Valeur', 'reprises_pertes_valeur_provisions': 'Reprise sur pertes de valeur et provisions',
   'resultat_operationnel': 'V-Resultat operationnel', 'produits_financiers': 'Produits financiers',
   'charges_financieres': 'Charges financieres', 'resultat_financier': 'VI-Resultat Financier',
   'resultat_ordinaire': 'VII-Resultat ordinaire', 'elements_extraordinaires_produits': 'Elements extraordinaires Produits',
   'elements_extraordinaires_charges': 'Elements extraordinaires Charges', 'resultat_extraordinaire': 'VIII-Resultat extraordinaire',
   'impots_exigibles_resultats': 'Impots exigibles sur resultats', 'impots_differes_resultats': 'Impots differes sur resultats',
   'resultat_net_exercice': 'RESULTAT NET DE L EXERCICE'}},
 'DECL': {'cols': ['valeur'], 'kinds': {'nif':'nif','raison_sociale':'texte','activite_principale':'texte','registre_commerce':'texte','adresse_siege':'texte','cac_cabinet':'texte','cac_nom':'texte','exercice_annee':'annee','annee_souscription':'annee'}, 'postes': {
   'nif': 'Numero d Identification Fiscale', 'raison_sociale': 'Designation de l entreprise',
   'activite_principale': 'Activite principale', 'registre_commerce': 'Registre de Commerce',
   'adresse_siege': 'Adresse siege social', 'cac_cabinet': 'Certification des comptes - Cabinet',
   'cac_nom': 'Certification des comptes - Nom CAC', 'exercice_annee': 'Resultat de l exercice - Annee',
   'annee_souscription': 'Annee de souscription', 'chiffre_affaires_global_ht': 'Chiffre d affaires global hors taxes',
   'resultat_comptable': 'Resultat comptable', 'resultat_fiscal': 'Resultat fiscal'}}
}
print('✅ Schémas V15 OK —', len(SCHEMAS), 'tableaux')

In [ ]:
# Classification par signature texte (V10C) — inchangée
TITLE_SIGNATURES = []
MAIN_SIGNATURES = [
    ('ACTIF', ['BILAN','ACTIF'], ['PASSIF']),
    ('PASSIF', ['BILAN','PASSIF'], []),
    ('TCR', ['COMPTE DE RESULTAT'], []),
    ('DECL', ['DECLARATION'], []),
]
def _norm(t):
    t = strip_accents(t or '').upper()
    t = re.sub(r'[^A-Z0-9/ ]+',' ', t)
    return ' '.join(t.split())
_MAIN_SIGS = [(c, [_norm(m) for m in ms], [_norm(f) for f in fs]) for c,ms,fs in MAIN_SIGNATURES]
def _match_signatures(t, sigs):
    out = []
    for c, musts, forb in sigs:
        if all(m in t for m in musts) and not any(f in t for f in forb):
            if c not in out: out.append(c)
    return out
def types_from_title(titre):
    brut = _norm(titre)
    if not brut: return []
    return _match_signatures(brut, _MAIN_SIGS)
PROMPT_CLASSIF = ('Page dun dossier fiscal algerien Serie G. Reponds UN seul mot : '
 'ACTIF si titre BILAN (ACTIF) ; PASSIF si BILAN (PASSIF) ; TCR si COMPTE DE RESULTAT ; DECL si page DECLARATION ; AUTRE sinon.')
RULES = [
 'REGLES: JSON valide uniquement, sans markdown, sans backticks.',
 'Aucune valeur inventee. Case vide ou absent: null. Illisible: null.',
 'Montants en nombres JSON sans separateurs de milliers.',
 'Montant entre parentheses = negatif: (1 553 799) devient -1553799.']
def build_prompt(types):
    L = ['Lis cette page d une liasse fiscale algerienne (imprime Serie G).',
         'Extrais en JSON strict les tableaux suivants, identifies par leur code.',
         'Structure: { type_page, entete: {entreprise, nif, exercice}, puis une cle par code de tableau present.',
         'IMPORTANT: recopie AUSSI les sous-totaux / postes groupes quand ils sont imprimes (ex Immobilisations corporelles, Immobilisations financieres, Creances et emplois assimiles, Disponibilites et assimiles). S ils sont absents, mets null.']
    for t in types:
        spec = SCHEMAS[t]
        L.append('--- Si ' + t + ' ---')
        L.append('Colonnes: ' + ', '.join(spec['cols']))
        for k, lab in spec['postes'].items():
            L.append(k + ' : ligne "' + lab + '"')
    L.extend(RULES)
    return chr(10).join(L)
print('✅ Prompts V15 OK')

In [ ]:
def norm_str(v):
    if v is None: return None
    s = re.sub(r'\s+',' ', str(v).strip())
    return s if s and s.lower() not in ('null','none','n/a') else None
def norm_montant(v):
    if v is None: return None
    if isinstance(v,(int,float)): return float(v)
    s = str(v).strip()
    neg = (s.startswith('(') and s.endswith(')')) or s.startswith('-')
    s = re.sub(r'[^0-9.,-]','', s)
    if not s: return None
    if s.count(',')==1 and '.' not in s: s = s.replace(',','.')
    elif ',' in s: s = s.replace(',','')
    elif s.count('.')>1: s = s.replace('.','')
    try: return -float(s) if neg else float(s)
    except Exception: return None
def norm_nif(v):
    s = norm_str(v)
    return re.sub('[^0-9]','', s) if s else None
def norm_annee4(v):
    s = norm_str(v) or ''
    m = re.findall(r'20\d{2}', s)
    return m[-1] if m else None
def norm_by_kind(v, kind):
    if kind=='texte': return norm_str(v)
    if kind=='nif': return norm_nif(v)
    if kind=='annee': return norm_annee4(v)
    return norm_montant(v)
def normalise_table(table, data):
    spec = SCHEMAS[table]
    kinds = spec.get('kinds', {})
    raw = (data or {}).get('postes') or {}
    out = {}
    for key in spec['postes']:
        vals = raw.get(key)
        vals = vals if isinstance(vals, dict) else {}
        for col in spec['cols']:
            out[key + '_' + col] = norm_by_kind(vals.get(col), kinds.get(key,'montant'))
    return out
def normalise_decl(data):
    raw = (data or {}).get('postes') or {}
    d = {}
    for key, kind in SCHEMAS['DECL']['kinds'].items():
        d[key] = norm_by_kind(raw.get(key), kind)
    for key in ['chiffre_affaires_global_ht','resultat_comptable','resultat_fiscal']:
        d[key] = norm_montant(raw.get(key))
    return d
print('✅ Normalisation OK')

In [ ]:
def build_page_object(page, tpage, data, rep, fichier):
    return {'page_id': 'p' + str(page['index']+1).zfill(3), 'index': page['index'],
            'fichier_source': fichier, 'type': tpage,
            'donnees': data or {}, 'tokens': (rep or {}).get('tokens_in',0) + (rep or {}).get('tokens_out',0)}
def fusion_tcr(pages_tcr):
    tcr = {}
    for d in pages_tcr:
        for k, v in normalise_table('TCR', d).items():
            if tcr.get(k) is None and v is not None: tcr[k] = v
    return tcr
print('✅ Construction pages OK')

In [ ]:
# PIPELINE (identique V13) — extrait ACTIF/PASSIF/TCR/DECL par page
deja = {f.stem for f in JSON_DIR.glob('*.json')}
a_traiter = [p for p in pdfs if p.stem not in deja]
log('A traiter: ' + str(len(a_traiter)))
for num, pdf_path in enumerate(a_traiter, start=1):
    try:
        pages = pdf_to_pages(pdf_path)
        actives = [p for p in pages if not is_blank(p['image'])]
        mini = [resize(p['image'], 600) for p in actives]
        reps1 = []
        for bs in range(0, len(mini), CLASSIF_BATCH_SIZE):
            reps1 += ask_batch(PROMPT_CLASSIF, mini[bs:bs+CLASSIF_BATCH_SIZE])
        utiles = [(p, r['text'].strip().upper()) for p, r in zip(actives, reps1) if r['text'].strip().upper() in ('ACTIF','PASSIF','TCR','DECL')]
        parsed = {}
        for bs in range(0, len(utiles), GPU_BATCH_SIZE):
            batch = utiles[bs:bs+GPU_BATCH_SIZE]
            reps = ask_batch(build_prompt([t for _, t in batch]), [p['image'] for p, _ in batch])
            for (p, t), rep in zip(batch, reps):
                parsed.setdefault(t, []).append(parse_json(rep['text']))
        result = {'fichier': pdf_path.name,
                  'ACTIF': normalise_table('ACTIF', parsed['ACTIF'][0]) if parsed.get('ACTIF') else {},
                  'PASSIF': normalise_table('PASSIF', parsed['PASSIF'][0]) if parsed.get('PASSIF') else {},
                  'TCR': fusion_tcr(parsed.get('TCR', [])),
                  'DECL': normalise_decl(parsed['DECL'][0]) if parsed.get('DECL') else {}}
        with open(JSON_DIR / (pdf_path.stem + '.json'), 'w', encoding='utf-8') as f:
            json.dump(result, f, ensure_ascii=False, indent=2, default=str)
        log('[' + str(num) + '/' + str(len(a_traiter)) + '] OK ' + pdf_path.name)
    except Exception as e:
        log('[' + str(num) + '] ERREUR ' + pdf_path.name + ' — ' + str(e))
log('✅ Extraction terminée')

In [ ]:
# ═══ CELLULE 12 — FORMULES V15 (postes groupés inclus) ═══
COLS_ACTIF = ['montant_brut','amortissements_provisions_pertes','net_n','net_n1']
COLS_PASSIF = ['n','n1']
FORMULES = [
 # ACTIF — sous-totaux groupés (V15)
 ('ACTIF','immobilisations_corporelles', COLS_ACTIF, [('+','terrains'),('+','batiments'),('+','autres_immobilisations_corporelles')]),
 ('ACTIF','immobilisations_financieres', COLS_ACTIF, [('+','titres_mis_en_equivalence'),('+','autres_participations_creances'),('+','autres_titres_immobilises'),('+','prets_actifs_financiers_non_courants'),('+','impots_differes_actif')]),
 ('ACTIF','creances_et_emplois_assimiles', COLS_ACTIF, [('+','clients'),('+','autres_debiteurs'),('+','impots_assimiles_actif'),('+','autres_creances_assimiles')]),
 ('ACTIF','disponibilites_et_assimiles', COLS_ACTIF, [('+','placements_financiers_courants'),('+','tresorerie_actif')]),
 ('ACTIF','total_actif_non_courant', COLS_ACTIF, [('+','ecarts_acquisition_goodwill'),('+','immobilisations_incorporelles'),('+','immobilisations_corporelles'),('+','immobilisations_en_concession'),('+','immobilisations_en_cours'),('+','immobilisations_financieres')]),
 ('ACTIF','total_actif_courant', COLS_ACTIF, [('+','stocks_encours'),('+','creances_et_emplois_assimiles'),('+','disponibilites_et_assimiles')]),
 ('ACTIF','total_general_actif', COLS_ACTIF, [('+','total_actif_non_courant'),('+','total_actif_courant')]),
 # PASSIF — totaux (conformes à vos égalités)
 ('PASSIF','total_capitaux_propres', COLS_PASSIF, [('+','capital_emis'),('+','capital_non_appele'),('+','primes_reserves'),('+','ecart_reevaluation'),('+','ecart_equivalence'),('+','resultat_net_passif'),('+','report_a_nouveau')]),
 ('PASSIF','total_passifs_non_courants', COLS_PASSIF, [('+','emprunts_dettes_financieres'),('+','impots_differes_provisionnes'),('+','autres_dettes_non_courantes'),('+','provisions_produits_avance')]),
 ('PASSIF','total_passifs_courants', COLS_PASSIF, [('+','fournisseurs_rattaches'),('+','impots_passif'),('+','autres_dettes'),('+','tresorerie_passif')]),
 ('PASSIF','total_general_passif', COLS_PASSIF, [('+','total_capitaux_propres'),('+','total_passifs_non_courants'),('+','total_passifs_courants')]),
]
CONTROLES = [
 ('Equilibre du bilan (N)', 'critique', [('+','ACTIF','total_general_actif','net_n')], [('+','PASSIF','total_general_passif','n')]),
 ('Equilibre du bilan (N-1)', 'critique', [('+','ACTIF','total_general_actif','net_n1')], [('+','PASSIF','total_general_passif','n1')]),
 ('Resultat net : TCR = passif (N)', 'critique', [('+','TCR','resultat_net_exercice','NET')], [('+','PASSIF','resultat_net_passif','n')]),
]
print('✅ Formules V15 OK —', len(FORMULES), 'formules,', len(CONTROLES), 'contrôles')

In [ ]:
# ═══ CELLULE 13 — MOTEUR DE COHÉRENCE V15 ═══
# Règle V15 : un contrôle n'est évalué QUE si la valeur déclarée (extraite) est
# présente ET que le recalcul est possible. Si la valeur groupée est absente
# (null), le contrôle est SAUTÉ : aucun écart null-vs-formule n'est affiché.
TOLERANCE_DA = 1.0
def _valeur(idx, tab, rc, col):
    vals = idx.get((tab, rc))
    if vals is None: return None
    if col in ('NET','NET_N1'):
        suf = ('n_credit','n_debit') if col=='NET' else ('n1_credit','n1_debit')
        c, d = vals.get(suf[0]), vals.get(suf[1])
        if c is None and d is None: return None
        return float(c or 0) - float(d or 0)
    v = vals.get(col)
    return float(v) if isinstance(v,(int,float)) and not isinstance(v,bool) else None
def _somme(idx, termes):
    total, vus = 0.0, 0
    for signe, tab, rc, col in termes:
        v = _valeur(idx, tab, rc, col)
        if v is not None:
            total += v if signe=='+' else -v; vus += 1
    return total if vus else None
def verifier_coherence(doc, tolerance=TOLERANCE_DA):
    idx = {}
    for tab in ('ACTIF','PASSIF','TCR'):
        bloc = doc.get(tab) or {}
        for cle_col, v in bloc.items():
            if '_' not in cle_col: continue
            rc, col = cle_col.rsplit('_', 1)
            # reconstitue (rc, col) même pour cols composées (n_debit...)
            for cc in ('n_debit','n_credit','n1_debit','n1_credit','montant_brut','amortissements_provisions_pertes','net_n','net_n1','n','n1'):
                if cle_col.endswith('_' + cc):
                    rc = cle_col[:-(len(cc)+1)]
                    idx.setdefault((tab, rc), {})[cc] = v
    res = []
    for tab, cible, cols, comps in FORMULES:
        for col in cols:
            declaree = _valeur(idx, tab, cible, col)
            recalc = _somme(idx, [(s, tab, rc, col) for s, rc in comps])
            if declaree is None or recalc is None:
                continue   # V15 : absent => pas d'écart affiché
            ecart = declaree - recalc
            res.append({'type':'formule','tableau':tab,'poste':cible,'colonne':col,
                        'valeur_extraite':declaree,'valeur_recalculee':recalc,
                        'ecart':round(ecart,2),
                        'statut':'coherent' if abs(ecart)<=tolerance else 'ecart_significatif'})
    for lib, grav, g, d in CONTROLES:
        a, b = _somme(idx, g), _somme(idx, d)
        if a is None or b is None:
            res.append({'controle':lib,'gravite':grav,'statut':'non_verifiable'})
            continue
        ecart = a - b
        res.append({'controle':lib,'gravite':grav,'valeur_a':round(a,2),'valeur_b':round(b,2),
                    'ecart':round(ecart,2),
                    'statut':'coherent' if abs(ecart)<=tolerance else 'ecart_significatif'})
    return res
print('✅ Moteur V15 OK')

In [ ]:
# ═══ CELLULE 14 — MAPPING EXCEL (sous-totaux ACTIF ajoutés) ═══
MAP_FEUILLES = {'ACTIF':'1. Bilan Actif','PASSIF':'2. Bilan Passif','TCR':'3. TCR'}
MAP_COLONNES = {'ACTIF':{'montant_brut':'B','amortissements_provisions_pertes':'C','net_n':'D','net_n1':'E'},
                'PASSIF':{'n':'B','n1':'C'}}
MAP_LIGNES = {'ACTIF': {
  'ecarts_acquisition_goodwill': (11,False), 'immobilisations_incorporelles': (12,False),
  'immobilisations_corporelles': (13,False), 'terrains': (14,False), 'batiments': (15,False),
  'autres_immobilisations_corporelles': (16,False), 'immobilisations_en_concession': (17,False),
  'immobilisations_en_cours': (18,False), 'immobilisations_financieres': (19,False),
  'titres_mis_en_equivalence': (20,False), 'autres_participations_creances': (21,False),
  'autres_titres_immobilises': (22,False), 'prets_actifs_financiers_non_courants': (23,False),
  'impots_differes_actif': (24,False), 'total_actif_non_courant': (25,True),
  'stocks_encours': (27,False), 'creances_et_emplois_assimiles': (28,False),
  'clients': (29,False), 'autres_debiteurs': (30,False), 'impots_assimiles_actif': (31,False),
  'autres_creances_assimiles': (32,False), 'disponibilites_et_assimiles': (33,False),
  'placements_financiers_courants': (34,False), 'tresorerie_actif': (35,False),
  'total_actif_courant': (36,True), 'total_general_actif': (37,True)},
 'PASSIF': {
  'capital_emis': (10,False), 'capital_non_appele': (11,False), 'primes_reserves': (12,False),
  'ecart_reevaluation': (13,False), 'ecart_equivalence': (14,False), 'resultat_net_passif': (15,False),
  'report_a_nouveau': (16,False), 'part_societe_consolidante': (17,False), 'part_minoritaires': (18,False),
  'total_capitaux_propres': (19,True), 'emprunts_dettes_financieres': (21,False),
  'impots_differes_provisionnes': (22,False), 'autres_dettes_non_courantes': (23,False),
  'provisions_produits_avance': (24,False), 'total_passifs_non_courants': (25,True),
  'fournisseurs_rattaches': (27,False), 'impots_passif': (28,False), 'autres_dettes': (29,False),
  'tresorerie_passif': (30,False), 'total_passifs_courants': (31,True), 'total_general_passif': (32,True)}}
print('✅ Mapping V15 OK')

In [ ]:
# ═══ CELLULE 15 — EXECUTION + RAPPORT ═══
import openpyxl
fichiers = sorted(JSON_DIR.glob('*.json'))
for jf in fichiers:
    doc = json.load(open(jf, encoding='utf-8'))
    rap = verifier_coherence(doc)
    doc['controles'] = rap
    nb_ec = sum(1 for r in rap if r.get('statut')=='ecart_significatif')
    nb_nv = sum(1 for r in rap if r.get('statut')=='non_verifiable')
    print()
    print('📄 ' + jf.name + ' | écarts:', nb_ec, '| non vérifiables:', nb_nv)
    for r in rap:
        if r.get('statut')=='ecart_significatif':
            print('   ❌ ' + str(r.get('poste') or r.get('controle')) + ' écart=' + str(r['ecart']))
print()
print('✅ Contrôles V15 terminés')